# `data_prep.ipynb`

Loads video and comment data, applies filtering (e.g., blocking unwanted terms), runs language detection in parallel, and calculates basic statistics. It uses NLTK resources for stopwords and tokenization, stores intermediate results periodically, and prepares final datasets for further analysis or modeling.

### Enhancements:

`@Guilherme O. S.`: 

- Added a keywords and exclude_words file at utils/*.txt. 
- Cleaned the code.
- Fixed the filter function.
- Parallelism and chunk saving.
- extended filter to the description.

---

In [ ]:
# import os
# import pandas as pd
# from glob import glob

# Example of how to concatenate CSV parts into a single file
# Replace `data_folder` with the actual folder path if needed
# data_folder = "raw_data"  # Replace with the real data folder path

# Example read parameters for CSV files
# read_csv_params = {
#     "on_bad_lines": "skip",
#     "encoding": "utf-8",
#     "sep": ",",
#     "lineterminator": "\n"
# }

# Base filenames to process (excluding files like `actual_date.csv`)
# files_base_names = [
#     "channels_info",
#     "comments_info",
#     "processed_videos",
#     "videos_info"
# ]

# For each base name, find matching parts, concatenate and save incrementally
# for base_name in files_base_names:
#     file_parts = sorted(glob(os.path.join(data_folder, f"{base_name}*.csv")))
#     if file_parts:
#         print(f"Processing files for: {base_name}")
#         output_file = os.path.join(data_folder, f"{base_name}.csv")
#         combined_df = pd.DataFrame()
#         for file_path in file_parts:
#             part_df = pd.read_csv(file_path, **read_csv_params)
#             combined_df = pd.concat([combined_df, part_df]).drop_duplicates()
#             combined_df.to_csv(output_file, index=False)
#             print(f"Partial file saved: {output_file}")
#         print(f"Final file saved: {output_file}")
#     else:
#         print(f"No files found for: {base_name}")

## Imports and Definitions

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from nltk.corpus import stopwords
import nltk
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist
import re
import pickle
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
from tqdm import tqdm
from langdetect import detect, LangDetectException
from typing import Set, List, Dict, Tuple
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing
import numpy as np
import os
import math
from pathlib import Path

nltk.download('stopwords')
nltk.download('punkt')

#==============================================================================
# Project paths and Constants
#==============================================================================

# Use the repository root (current working directory) as base
PROJECT_ROOT = os.getcwd()
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, 'raw_data')
CLEANED_DATA_DIR = os.path.join(PROJECT_ROOT, 'cleaned_data')
FIGS_DIR = os.path.join(PROJECT_ROOT, 'figs')
UTILS_DIR = os.path.join(PROJECT_ROOT, 'utils')

# Ensure output folders exist
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(CLEANED_DATA_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)
os.makedirs(UTILS_DIR, exist_ok=True)

# Files (use these variables throughout the notebook)
VIDEOS_FILE = os.path.join(RAW_DATA_DIR, 'videos_info.csv')
COMMENTS_FILE = os.path.join(RAW_DATA_DIR, 'comments_info.csv')
FILTERED_VIDEOS_FILE = os.path.join(CLEANED_DATA_DIR, 'filtered_videos.csv')
EXCLUDED_VIDEOS_FILE = os.path.join(CLEANED_DATA_DIR, 'excluded_videos.csv')
LANGUAGE_DETECT_STATE_FILE = os.path.join(PROJECT_ROOT, 'dict_language_detected.pkl')
REMAINING_UNPROCESSED_FILE = os.path.join(PROJECT_ROOT, 'remaining_unprocessed_ids.txt')
FINAL_COMMENTS_FILE = os.path.join(CLEANED_DATA_DIR, 'final_commments_match.csv')
SAMPLED_COMMENTS_FILE = os.path.join(PROJECT_ROOT, 'Final_Comments_Sampled.csv')

# Language detection outputs
COMMENTS_LANG_FILE = os.path.join(CLEANED_DATA_DIR, 'comments_info_language_roBERTa_substring.csv')
COMMENTS_LANG_FILE_SAMPLE = os.path.join(CLEANED_DATA_DIR, 'comments_info_language_roBERTa_substring_sample.csv')

# Utils files
KEYWORDS_FILE_PATH = os.path.join(UTILS_DIR, 'keywords.txt')
EXCLUDE_WORDS_FILE_PATH = os.path.join(UTILS_DIR, 'exclude_words.txt')
MANDATORY_KEYWORDS_FILE_PATH = os.path.join(UTILS_DIR, 'mandatory.txt')

# Date range
START_DATE = '2020-01-01'
END_DATE = '2025-12-31'

# If you don't have the stopwords downloaded, set to True
DOWNLOAD_STOPWORDS = False

# SAMPLE MODE CONSTANTS
USE_SAMPLE_MODE = False          # Set to False to process all data
SAMPLE_SIZE = 1000          
N_COMMENTS = 20                # Sample size for general comments per month
N_COMMENTS_FROM_KEYWORD = 10    # Sample size for keyword-containing comments per month

# LANGUAGE DETECTION CONSTANTS
NUM_WORKERS = 1
SAVE_INTERVAL = 20000           # Save after this many comments
BATCH_SIZE = 1024                # Batch size for processing

---

## Data Loading and Filtering

### Utils/Functions definitions

In [ ]:
mandatory_terms = [] 
blocked_terms = []
focused_terms = []

def load_stopwords(download: bool = False) -> Set[str]:
    """Load and combine stopwords from different languages."""
    if download:
        nltk.download('stopwords')
        nltk.download('punkt')
    
    stopwords_english = set(stopwords.words('english'))
    stopwords_french = set(stopwords.words('french'))
    custom_stopwords = set()
    return stopwords_english.union(stopwords_french, custom_stopwords)

def load_terms(file_path: str, stopwords: Set[str] = None) -> List[str]:
    """Load terms from a file and optionally filter out stopwords."""
    with open(file_path, 'r') as file:
        terms = file.read().strip().split(',')
    if stopwords:
        return [word for word in terms if word not in stopwords]
    return terms

def filter_text(text: str) -> bool:
    """Filter text based on mandatory, blocked, and focused terms."""
    if not isinstance(text, str):  # Handle NaN or non-string cases
        return False

    text = text.lower()
    
    def has_word(word: str, text: str) -> bool:
        """Check if a word exists in text as a whole word."""
        pattern = r'\b' + re.escape(word.lower()) + r'\b'
        return bool(re.search(pattern, text))
    
    # Check mandatory terms first
    for term in mandatory_terms:
        if has_word(term, text):
            return True
            
    # Check blocked terms
    for term in blocked_terms:
        if has_word(term, text):
            return False
            
    # # Check focused terms
    # for term in focused_terms:
    #     if has_word(term, text):
    #         return True
            
    return False

def clean_titles(titles: pd.Series, stopwords: Set[str]) -> pd.Series:
    """Remove stopwords from titles."""
    return titles.apply(lambda x: ' '.join([word for word in x.split() if word.lower() not in stopwords]))

def plot_word_cloud(titles: pd.Series, file_name: str):
    """Generate and save word cloud visualization."""
    wordcloud = WordCloud(width=1920, height=
                          80, background_color='white').generate(' '.join(titles))
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.savefig(file_name, bbox_inches='tight')
    plt.show()

### Word Cloud Plotting and saving

In [ ]:
if DOWNLOAD_STOPWORDS:
    nltk.download('stopwords')
    nltk.download('punkt')

# Load videos using the centralized path variable
videos_df = pd.read_csv(VIDEOS_FILE)

# Remove duplicates and filter by date
video_df = videos_df.drop_duplicates(subset=['video_id'])
video_df['published_at'] = video_df['published_at'].str.slice(0, 10)
video_df = video_df[(video_df['published_at'] >= START_DATE) & 
                        (video_df['published_at'] <= END_DATE)]

# Load stopwords and terms
final_stopwords = load_stopwords(DOWNLOAD_STOPWORDS)

# Define the terms for filtering
mandatory_terms = load_terms(MANDATORY_KEYWORDS_FILE_PATH, final_stopwords)
focused_terms = load_terms(KEYWORDS_FILE_PATH, final_stopwords)
blocked_terms = load_terms(EXCLUDE_WORDS_FILE_PATH)

# Filter videos using the filter_text function
filtered_videos = video_df[video_df['title'].apply(filter_text)]

# Get videos that didn't pass the filter
excluded_videos = video_df[~video_df['title'].apply(filter_text)]       # title
_ = video_df[~video_df['description'].apply(filter_text)] # description (not assigned)

cleaned_filtered_titles = clean_titles(filtered_videos['title'], final_stopwords)
cleaned_excluded_titles = clean_titles(excluded_videos['title'], final_stopwords)

plot_word_cloud(cleaned_filtered_titles, os.path.join(FIGS_DIR, 'cleaned_filtered_titles.png'))
plot_word_cloud(cleaned_excluded_titles, os.path.join(FIGS_DIR, 'cleaned_excluded_titles.png'))

filtered_videos.to_csv(FILTERED_VIDEOS_FILE, index=False)
excluded_videos.to_csv(EXCLUDED_VIDEOS_FILE, index=False)

print(f"Total number of videos: {len(video_df)}")
print(f"Number of filtered videos: {len(filtered_videos)}")

---

## Comments Language Detection

### Utils/Functions definitions

In [ ]:
def save_state(language_dict: Dict, filename: str = LANGUAGE_DETECT_STATE_FILE) -> None:
    """Save the language detection state to a file."""
    with open(filename, "wb") as f:
        pickle.dump(language_dict, f)

def load_state(filename: str = LANGUAGE_DETECT_STATE_FILE) -> Dict:
    """Load the language detection state from a file."""
    try:
        with open(filename, "rb") as f:
            return pickle.load(f)
    except (FileNotFoundError, pickle.UnpicklingError, Exception):
        return {}

def init_model():
    """Initialize the model and tokenizer."""
    model_ckpt = "papluca/xlm-roberta-base-language-detection"
    tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AutoModelForSequenceClassification.from_pretrained(model_ckpt).to(device)
    return tokenizer, model, device

def process_batch(batch_data: Tuple[List[str], List[str]], model_ckpt: str) -> Dict[str, str]:
    """Process a batch of comments using the language detection model."""
    batch_ids, batch_comments = batch_data
    
    # Initialize model for this process
    tokenizer, model, device = init_model()
    results = {}
    
    try:
        inputs = tokenizer(batch_comments, padding=True, truncation=True, 
                         max_length=256, return_tensors="pt").to(device)
        
        with torch.no_grad():
            logits = model(**inputs).logits
        
        preds = torch.softmax(logits, dim=-1).argmax(dim=1).tolist()
        
        for cid, idx in zip(batch_ids, preds):
            results[cid] = model.config.id2label[idx]
            
        # Clean up
        del inputs, logits
        torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"Error in batch processing: {e}")
        
    return results

def chunks(lst: List, n: int):
    """Split a list into n-sized chunks."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

In [ ]:
dict_comment_language = load_state()

# Read and filter comments using centralized path variables
print("Loading and filtering comments...")
df_comments = pd.read_csv(COMMENTS_FILE, 
                         on_bad_lines='skip', 
                         encoding='utf-8', 
                         sep=',', 
                         lineterminator='\n')

df_comments['published_at'] = pd.to_datetime(df_comments['published_at'], errors='coerce')
df_comments = df_comments.dropna(subset=['published_at']).drop_duplicates()

videos_with_keyword = pd.read_csv(FILTERED_VIDEOS_FILE)

# Basic preprocessing
df_comments['comment'] = df_comments['comment'].astype(str)
df_comments['published_at'] = pd.to_datetime(df_comments['published_at'])
df_comments = df_comments[
    (df_comments['published_at'] >= START_DATE) & 
    (df_comments['published_at'] <= END_DATE)
]
df_comments = df_comments[
    df_comments['video_id'].isin(set(videos_with_keyword['video_id']))
]

# Sample mode handling
if USE_SAMPLE_MODE:
    print(f"Using sample mode with {SAMPLE_SIZE} comments")
    df_comments = df_comments.sample(n=min(SAMPLE_SIZE, len(df_comments)))

# Prepare data for processing
dict_comments_to_detect_language = dict(
    zip(df_comments["comment_id"].astype(str), df_comments["comment"])
)

# Ensure keys are strings
dict_comment_language = {str(k): v for k, v in dict_comment_language.items()}

# Identify unprocessed comments
unprocessed_comment_ids = [
    cid for cid in dict_comments_to_detect_language 
    if cid not in dict_comment_language
]

if not unprocessed_comment_ids:
    print("No unprocessed comments found.")
else:
    print(f"Processing {len(unprocessed_comment_ids)} comments...")
    
    # Prepare batches for parallel processing
    batches = list(chunks(unprocessed_comment_ids, BATCH_SIZE))
    batch_data = [
        ([cid for cid in batch],
         [dict_comments_to_detect_language[cid] for cid in batch])
        for batch in batches
    ]
    
    # Process batches in parallel
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        future_to_batch = {
            executor.submit(
                process_batch, 
                batch, 
                "papluca/xlm-roberta-base-language-detection"
            ): i 
            for i, batch in enumerate(batch_data)
        }
        
        # Process results as they complete
        processed_count = 0
        for future in tqdm(as_completed(future_to_batch), 
                         total=len(future_to_batch),
                         desc="Processing batches"):
            try:
                batch_results = future.result()
                dict_comment_language.update(batch_results)
                processed_count += len(batch_results)
                
                # Save state periodically
                if processed_count >= SAVE_INTERVAL:
                    save_state(dict_comment_language)
                    processed_count = 0
                    
            except Exception as e:
                print(f"Batch processing failed: {e}")

    # Final save
    save_state(dict_comment_language)
    
    # Save remaining unprocessed IDs if any
    remaining_unprocessed_ids = [
        cid for cid in unprocessed_comment_ids 
        if cid not in dict_comment_language
    ]
    if remaining_unprocessed_ids:
        with open("remaining_unprocessed_ids.txt", "w") as f:
            for cid in remaining_unprocessed_ids:
                f.write(f"{cid}\n")

# Update DataFrame with language information
df_comments['comment_id'] = df_comments['comment_id'].astype(str)
df_comments['language'] = df_comments['comment_id'].map(dict_comment_language)

# Save results
output_filename = (
    "cleaned_data/comments_info_language_roBERTa_substring_sample.csv"
    if USE_SAMPLE_MODE else
    "cleaned_data/comments_info_language_roBERTa_substring.csv"
)
df_comments.to_csv(output_filename, index=False)

print(f"Processing complete. Results saved to {output_filename}")

#### **Language Tool: Second language check**

RoBERTa didn't perform very well on noisy comments. For example: Kkkk Okkk. Merciiiii!
It's worth adding a second attempt. If it doesn't pass any of them, it won't be useful for our model either
because the noise level is really very high! This is common in social networks!!!
Therefore, I made a second pass for those that were not considered as FR (French) by LangDetect.

With this, we assume that if the comment is considered FR by at least one of the APIs, it is included in the analysis.

In [ ]:
def detect_language(text: str) -> str:
    """Detect language of a given text with proper error handling."""
    try:
        text = str(text).strip()
        if not text or len(text) < 3:
            return 'unidentified'
        return detect(text)
    except LangDetectException:
        return 'unidentified'
    except Exception as e:
        print(f"Error in language detection: {str(e)[:100]}")
        return 'unidentified'

def process_batch(batch_data):
    """Process a batch of comments for language detection."""
    return [detect_language(text) for text in batch_data]

def process_comments_language(input_file: str) -> pd.DataFrame:
    """Process comments file for language detection using parallel processing."""
    print("Loading comments data...")
    df_comments = pd.read_csv(
        input_file, 
        on_bad_lines='skip', 
        encoding='utf-8', 
        sep=',', 
        lineterminator='\n'
    )
    
    if USE_SAMPLE_MODE:
        print(f"Using sample mode with {SAMPLE_SIZE} comments")
        df_comments = df_comments.sample(n=min(SAMPLE_SIZE, len(df_comments)))
    
    # Process dates and clean data
    df_comments['published_at'] = pd.to_datetime(df_comments['published_at'])
    df_comments = df_comments.dropna(subset=['published_at'])
    mask = df_comments['language'] != 'pt'
    comments_to_process = df_comments[mask]['comment'].values
    
    # Determine optimal batch size and number of workers
    num_cpus = os.cpu_count() - 2 
    batch_size = math.ceil(len(comments_to_process) / (num_cpus * 100))  # 4 batches per CPU
    num_batches = math.ceil(len(comments_to_process) / batch_size)
    
    
    detected_languages = []
    with ProcessPoolExecutor(max_workers=num_cpus) as executor:
        batches = [
            comments_to_process[i:i + batch_size] 
            for i in range(0, len(comments_to_process), batch_size)
        ]
        
        futures = list(tqdm(
            executor.map(process_batch, batches),
            total=num_batches,
            desc="Processing batches"
        ))
        
        detected_languages = [lang for batch_result in futures for lang in batch_result]
    df_comments.loc[mask, 'language_langdetect'] = detected_languages
    
    return df_comments

def filter_portuguese_comments(df: pd.DataFrame, output_file: str) -> pd.DataFrame:
    """Filter Portuguese comments and save to file."""
    df_comments_pt = df[
        (df['language'] == 'pt') | 
        (df['language_langdetect'] == 'pt')
    ]
    
    df_comments_pt.to_csv(output_file, index=False)
    
    return df_comments_pt

def process_keyword_sampling(
    df_comments: pd.DataFrame,
    df_videos: pd.DataFrame,
    keywords: List[str]
) -> pd.DataFrame:
    """Process and sample comments based on keywords."""
    # Clean up dates and comment text
    df_comments['published_at'] = pd.to_datetime(df_comments['published_at'], errors='coerce')
    df_comments.dropna(subset=['published_at'], inplace=True)
    df_comments['comment'] = df_comments['comment'].astype(str)
    
    # Define keyword check function
    def contains_keyword(text: str) -> bool:
        text = text.lower()
        return any(keyword.lower() in text for keyword in keywords)
    
    # Apply keyword check
    df_comments['contains_keyword'] = df_comments['comment'].apply(contains_keyword)
    
    # Sample comments
    sample_all = df_comments.groupby(
        df_comments['published_at'].dt.to_period("M"),
        group_keys=False
    ).apply(lambda x: x.sample(min(len(x), N_COMMENTS)))
    
    df_filtered = df_comments[df_comments['contains_keyword']]
    sample_filtered = df_filtered.groupby(
        df_filtered['published_at'].dt.to_period("M"),
        group_keys=False
    ).apply(lambda x: x.sample(min(len(x), N_COMMENTS_FROM_KEYWORD)))
    
    # Combine and merge samples with video data
    sample_combined = pd.concat([sample_all, sample_filtered])
    sample_combined = sample_combined[['video_id', 'comment_id', 'published_at', 'comment']]
    sample_combined = pd.merge(sample_combined, df_videos, on='video_id', how='left')
    sample_combined = sample_combined.sample(frac=1).reset_index(drop=True)
    
    return sample_combined

In [ ]:
# Process initial language detection
input_file_for_lang = COMMENTS_LANG_FILE if not USE_SAMPLE_MODE else COMMENTS_LANG_FILE_SAMPLE
df_comments = process_comments_language(input_file_for_lang)
    
# Filter Portuguese comments and save using centralized path
df_comments_pt = filter_portuguese_comments(
    df_comments,
    FINAL_COMMENTS_FILE
)
    
df_videos = pd.read_csv(FILTERED_VIDEOS_FILE)
    
# Calculate statistics
stats = {
        'unique_channels': df_videos['channel_id'].nunique(),
        'unique_videos': df_videos['video_id'].nunique(),
        'total_comments': len(df_comments_pt),
        'unique_users': df_comments_pt['author_channel_id'].nunique()
}
    
print("\nStatistics:")
for key, value in stats.items():
    print(f"{key}: {value}")
    
sample_combined = process_keyword_sampling(df_comments_pt, df_videos, mandatory_terms)
    
# Save results
# print(sample_combined.columns)
sample_combined[['title', 'comment']].to_csv(SAMPLED_COMMENTS_FILE, index=False)
# sample_combined[sample_combined['contains_keyword']]['comment'].to_csv("Teste.csv", index=False)

---

# Unit Tests

In [ ]:
import pandas as pd

# Load data
df_comments = pd.read_csv(FINAL_COMMENTS_FILE)
df_comments['published_at'] = pd.to_datetime(df_comments['published_at'], errors='coerce')

# Filter rows that could not be converted to datetime
df_comments.dropna(subset=['published_at'], inplace=True)
df_comments['hour'] = df_comments['published_at'].dt.hour

# Filter comments from 1am to 5am (01:00 - 05:00) and 1pm to 5pm (13:00 - 17:00)
df_am = df_comments[(df_comments['hour'] >= 1) & (df_comments['hour'] < 5)]
df_pm = df_comments[(df_comments['hour'] >= 13) & (df_comments['hour'] < 17)]

def calculate_stats(df):
    author_comment_counts = df['author'].value_counts()
    video_comment_counts = df.groupby('author')['video_id'].nunique()
    return {
        'total_comments': len(df),
        'mean_comments': author_comment_counts.mean(),
        'std_comments': author_comment_counts.std(),
        'median_comments': author_comment_counts.median(),
        'q25_comments': author_comment_counts.quantile(0.25),
        'q75_comments': author_comment_counts.quantile(0.75),
        'q99_comments': author_comment_counts.quantile(0.99),
        'mean_videos': video_comment_counts.mean(),
        'std_videos': video_comment_counts.std(),
        'median_videos': video_comment_counts.median(),
        'q25_videos': video_comment_counts.quantile(0.25),
        'q75_videos': video_comment_counts.quantile(0.75),
        'q99_videos': video_comment_counts.quantile(0.99),
    }

# Calculate statistics for 1am-5am comments
stats_am = calculate_stats(df_am)
print("\nGeneral statistics for the 1am-5am group:")
print(stats_am)

# Calculate statistics for 1pm-5pm comments
stats_pm = calculate_stats(df_pm)
print("\nGeneral statistics for the 1pm-5pm group:")
print(stats_pm)